# 06 · Number of image components

Do the conclusions depend on how many principal components are kept per image type?

In the data pipeline (`notebooks/01_data_pipeline/02_pca_per_image_new_method`) one PCA was fitted per image
type on the raw HOG features (8,100 per image, no scaling). For each type the smallest number of components
reaching a target share of variance τ was found, and the largest of these was used for all five types, so that
every image block has the same size. That PCA used all 416 patients and τ = 0.592 (110 components).

Here the same procedure is repeated on the 216 analysis patients for τ = 0.50, 0.592 and 0.90 (a lower and a
higher threshold around the one used), and every configuration with images (I1, I2, M1, M2,
M3) is refitted at k = 4. The clinical configurations do not use images and do not change.

In [1]:
import json

%run -i ../src/setup.ipynb

from sklearn.decomposition import PCA

CMP = cfg.RESULTS_DIR / "02_compare_configurations"
OUT = cfg.RESULTS_DIR / "06_pca_robustness"
OUT.mkdir(parents=True, exist_ok=True)

selection = json.loads((CMP / "selection.json").read_text())
K = selection["k"]
df = load_analysis_data()
main_psi = pd.read_csv(CMP / "psi_table.csv", index_col=0)
TAUS = [0.50, 0.592, 0.90]
IMAGE_CONFIGS = ["I1", "I2", "I2-b", "M1", "M2", "M3"]

## 1. Raw HOG features of the 216 patients

In [2]:
hog = pd.read_csv(cfg.DATA_DIR / "raw_hog_features_per_image.csv")
hog = hog.set_index("patient_id").loc[df["patient_id"]]
print("HOG matrix:", hog.shape, "| patients in the same order as the analysis sample:",
      (hog.index == df["patient_id"]).all())

HOG matrix: (216, 40500) | patients in the same order as the analysis sample: True


## 2. PCA per image type for each τ

In [3]:
blocks = {t: hog[[c for c in hog.columns if c.startswith(f"{t}_HOG_")]].to_numpy(float) for t in cfg.IMAGE_TYPES}

rows, pcs_by_tau = [], {}
for tau in TAUS:
    needed = {t: PCA(n_components=tau, random_state=42).fit(X).n_components_ for t, X in blocks.items()}
    p = max(needed.values())
    parts = [df[["patient_id"]].reset_index(drop=True)]
    for t, X in blocks.items():
        Z = PCA(n_components=p, random_state=42).fit_transform(X)
        parts.append(pd.DataFrame(Z, columns=[f"{t}_PC{i + 1}" for i in range(p)]))
    pcs_by_tau[tau] = pd.concat(parts, axis=1)
    rows.append({"τ": tau, **{f"min comps {t}": n for t, n in needed.items()}, "components used per type (p)": p})
components = pd.DataFrame(rows).set_index("τ")
components.to_csv(OUT / "components_per_tau.csv")
components

,min comps cor,min comps gfc_sag,min comps gfc_tra,min comps masked_tra,min comps sbj_sag,components used per type (p)
τ,,,,,,
0.500,52,48,48,55,18,55
0.592,70,66,67,72,32,72
0.900,163,159,160,159,138,163


## 3. Refitting the image configurations

M3 again takes the distance of whichever of M1 and M2 has the higher PSI.

In [4]:
results = {}
for tau, pcs in pcs_by_tau.items():
    d = with_image_pcs(df, pcs)
    col = {}
    for name in ["I1", "I2", "I2-b", "M1", "M2"]:
        col[name] = psi_column(d, fit_configuration(name, K, d)["labels"], name)
    m3_metric = "relms" if col["M1"]["PSI (mean)"] >= col["M2"]["PSI (mean)"] else "ggower"
    col["M3"] = psi_column(d, fit_configuration("M3", K, d, metric=m3_metric)["labels"], "M3")
    results[tau] = (pd.DataFrame(col), m3_metric)
    print(f"τ = {tau}: done (M3 uses {m3_metric})")

agg = pd.DataFrame({"main (416 pts, 110 PCs)": main_psi.loc["PSI (mean)"]})
for tau, (tab, _) in results.items():
    p = components.loc[tau, "components used per type (p)"]
    agg[f"τ = {tau} (216 pts, {p} PCs)"] = tab.loc["PSI (mean)"]
agg = agg.reindex(["C1", "C1-b", "C2"] + IMAGE_CONFIGS)
for c in ["C1", "C1-b", "C2"]:
    agg.loc[c] = agg.loc[c, agg.columns[0]]
agg.to_csv(OUT / "aggregate_psi_by_tau.csv")
print("Aggregate PSI (clinical configurations do not use images and are identical in every column):")
agg.round(3)

τ = 0.5: done (M3 uses ggower)


τ = 0.592: done (M3 uses ggower)


τ = 0.9: done (M3 uses ggower)
Aggregate PSI (clinical configurations do not use images and are identical in every column):


,"main (416 pts, 110 PCs)","τ = 0.5 (216 pts, 55 PCs)","τ = 0.592 (216 pts, 72 PCs)","τ = 0.9 (216 pts, 163 PCs)"
C1,0.434,0.434,0.434,0.434
C1-b,0.497,0.497,0.497,0.497
C2,0.367,0.367,0.367,0.367
I1,0.213,0.221,0.226,0.202
I2,0.218,0.256,0.225,0.231
I2-b,0.119,0.163,0.117,0.114
M1,0.399,0.341,0.389,0.391
M2,0.400,0.406,0.403,0.418
M3,0.350,0.369,0.394,0.404


In [5]:
print("Rank of each configuration (1 = highest PSI):")
display(agg.rank(ascending=False).astype(int))
for tau, (tab, _) in results.items():
    print(f"\nPer-variable PSI, τ = {tau}")
    display(format_psi_table(tab))
    tab.to_csv(OUT / f"psi_table_tau{tau}.csv")

Rank of each configuration (1 = highest PSI):


,"main (416 pts, 110 PCs)","τ = 0.5 (216 pts, 55 PCs)","τ = 0.592 (216 pts, 72 PCs)","τ = 0.9 (216 pts, 163 PCs)"
C1,2,2,2,2
C1-b,1,1,1,1
C2,5,5,6,6
I1,8,8,7,8
I2,7,7,8,7
I2-b,9,9,9,9
M1,4,6,5,5
M2,3,3,3,3
M3,6,4,4,4



Per-variable PSI, τ = 0.5


,I1,I2,I2-b,M1,M2,M3
Age,0.234 (S),0.310 (S),0.190 (S),0.120 (N),0.450 (M),0.404 (M)
MMSE,0.158 (S),0.203 (S),0.170 (S),0.220 (S),0.263 (S),0.200 (S)
nWBV,0.310 (S),0.406 (M),0.218 (S),0.101 (N),0.560 (L),0.495 (L)
eTIV,0.325 (S),0.329 (S),0.100 (N),0.383 (M),0.403 (M),0.419 (M)
CDR,0.152 (S),0.191 (S),0.086 (N),0.164 (S),0.245 (S),0.167 (S)
Educ,0.163 (S),0.143 (N),0.219 (S),0.584 (L),0.494 (L),0.462 (M)
SES,0.138 (N),0.142 (N),0.184 (S),0.701 (L),0.445 (M),0.432 (M)
sex,0.285 (S),0.320 (S),0.139 (N),0.458 (M),0.386 (M),0.373 (M)
PSI (mean),0.221 (S),0.256 (S),0.163 (S),0.341 (M),0.406 (M),0.369 (M)



Per-variable PSI, τ = 0.592


,I1,I2,I2-b,M1,M2,M3
Age,0.282 (S),0.306 (S),0.042 (N),0.191 (S),0.335 (M),0.386 (M)
MMSE,0.151 (S),0.199 (S),0.118 (N),0.370 (M),0.278 (S),0.194 (S)
nWBV,0.353 (M),0.359 (M),0.070 (N),0.303 (S),0.357 (M),0.505 (L)
eTIV,0.356 (M),0.305 (S),0.104 (N),0.400 (M),0.542 (L),0.469 (M)
CDR,0.147 (N),0.159 (S),0.049 (N),0.259 (S),0.201 (S),0.209 (S)
Educ,0.149 (S),0.124 (N),0.218 (S),0.610 (L),0.511 (L),0.522 (L)
SES,0.135 (N),0.139 (N),0.208 (S),0.685 (L),0.537 (L),0.469 (M)
sex,0.232 (S),0.206 (S),0.127 (N),0.290 (S),0.464 (M),0.397 (M)
PSI (mean),0.226 (S),0.225 (S),0.117 (N),0.389 (M),0.403 (M),0.394 (M)



Per-variable PSI, τ = 0.9


,I1,I2,I2-b,M1,M2,M3
Age,0.235 (S),0.365 (M),0.049 (N),0.095 (N),0.440 (M),0.376 (M)
MMSE,0.218 (S),0.194 (S),0.127 (N),0.354 (M),0.265 (S),0.229 (S)
nWBV,0.266 (S),0.412 (M),0.084 (N),0.351 (M),0.540 (L),0.474 (M)
eTIV,0.231 (S),0.168 (S),0.082 (N),0.341 (M),0.392 (M),0.398 (M)
CDR,0.168 (S),0.162 (S),0.047 (N),0.254 (S),0.232 (S),0.218 (S)
Educ,0.090 (N),0.255 (S),0.205 (S),0.684 (L),0.543 (L),0.604 (L)
SES,0.117 (N),0.189 (S),0.192 (S),0.755 (L),0.540 (L),0.543 (L)
sex,0.290 (S),0.100 (N),0.122 (N),0.295 (S),0.392 (M),0.395 (M)
PSI (mean),0.202 (S),0.231 (S),0.114 (N),0.391 (M),0.418 (M),0.404 (M)


## 4. Image distances with more components

With more components, the image blocks contain more low-variance, mostly noisy directions, and pairwise
distances become more alike. The table shows the relative spread (coefficient of variation) of the distances in
each block of M2 and how close M2's partition is to C1's.

In [6]:
from sklearn.metrics import adjusted_rand_score

c1 = fit_configuration("C1", K, df)["labels"]
iu = np.triu_indices(len(df), 1)
rows = []
for tau, pcs in pcs_by_tau.items():
    d = with_image_pcs(df, pcs)
    r = fit_configuration("M2", K, d)
    dist = [np.asarray(D, float) for D in r["model"].dist_output[1] if np.asarray(D).sum() > 0]
    cv = [np.std(D[iu]) / np.mean(D[iu]) for D in dist]
    rows.append({"τ": tau, "PCs per type": components.loc[tau, "components used per type (p)"],
                 "mean CV, image blocks": np.mean(cv[:5]), "CV, clinical block": cv[5],
                 "ARI(M2, C1)": adjusted_rand_score(r["labels"], c1)})
concentration = pd.DataFrame(rows).set_index("τ").round(3)
concentration.to_csv(OUT / "distance_concentration_M2.csv")
concentration

,PCs per type,"mean CV, image blocks","CV, clinical block","ARI(M2, C1)"
τ,,,,
0.500,55,0.134,0.382,0.221
0.592,72,0.120,0.382,0.249
0.900,163,0.081,0.382,0.209


## 5. Interpretation

- The conclusions do not depend on the number of components: at every threshold C1-b has the highest PSI (0.497),
  C1 the second highest (0.434), and the image-only configurations the lowest (I1 and I2 between 0.20 and 0.26,
  I2-b between 0.11 and 0.16).
- The configurations with images and clinical variables stay between 0.34 and 0.42, always below C1.
- Fitting the PCA on the 216 patients instead of all 416 changes the PSI of the image configurations only a little
  (compare the main column with τ = 0.592, now 72 components).
- With more components the image distances become more uniform (spread from about 0.13 to 0.08), but the mixed
  partitions stay about equally close to C1 (ARI 0.21 to 0.25).
- The main analysis keeps 110 components (τ = 0.592), which lies between 0.50 and 0.90; both give the same
  conclusions.